# **SETUP**

In [1]:
import pathlib
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

xgb = XGBClassifier(enable_categorical=True, n_jobs=-1)

# **OOF PREDICTIONS**

In [8]:
scores = []
oof = pd.Series(index=train.index, dtype=float, name="oof")

X = train.drop(columns=["PitNextLap"])
y = train["PitNextLap"]

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    xgb.fit(X[~is_val], y[~is_val])

    oof[is_val] = xgb.predict_proba(X[is_val])[:, 1]
    scores.append(roc_auc_score(y[is_val], oof[is_val]))

print(scores)
print("OOF ROC-AUC:", roc_auc_score(y, oof))

[0.9388354709320553, 0.9373572749297812, 0.9361778072579173, 0.934834001772325, 0.93705195416172]
OOF ROC-AUC: 0.936852697436653


# **TEST PREDICTIONS**

In [ ]:
xgb.fit(X, y)
preds = pd.Series(xgb.predict_proba(test)[:, 1], index=test.index, dtype=float, name="preds")

# **EXPORT**

In [21]:
oof.to_frame().to_parquet(PROJECT_ROOT / "data" / "oof" / "001-oof.parquet")
preds.to_frame().to_parquet(PROJECT_ROOT / "data" / "preds" / "001-preds.parquet")